# 04 · Визуальные представления, CLIP и текст → рамка → маска

Практикум к лекции 5. **Два занятия по 90 минут**, затем оформление отчёта (около 60–90 минут).
Различайте: (1) учебные численные матрицы, (2) реальный инференс замороженных моделей,
(3) истинную разметку данных. Ни один учебный массив ниже не обозначен как результат CLIP.

Для каждого стенда заполните четыре строки: **прогноз → изменённый параметр → наблюдение с числом → объяснение**.
Сначала меняйте один параметр. Ошибочный прогноз полезен, если указан механизм ошибки.
Исходный опыт каждого стенда выполняется один раз при открытии. Повторный дорогой CLIP/DINO/SAM запускается кнопкой.
DINO сохраняет logits для повторных порогов, а SAM повторно использует признаки того же изображения при смене рамки.

Ноутбук самодостаточен: весь код, небольшие метаданные и тесты встроены. Фотографии и веса скачиваются
в пользовательский кэш с SHA-256 и не должны входить в сдаваемый архив. Первый запуск: примерно 1.6 GiB весов + 54 MB Penn–Fudan;
нужно около 4 GB свободного места и 6–8 GB RAM.
Для CLIP, Grounding DINO и SAM автоматически выбирается CUDA, когда она доступна PyTorch; иначе CPU.
Выбранное устройство выводится в ячейке помощников. Учебные стенды с NumPy остаются на CPU.
Если видеокарта есть, а вывод показывает CPU, проверьте `torch.cuda.is_available()` и установку PyTorch с поддержкой CUDA.
Для ручного выбора задайте `VISUAL_LAB_DEVICE=cpu` или `VISUAL_LAB_DEVICE=cuda` до запуска ячейки помощников.
Оценки времени — ориентир; фактические измерения появятся в выводе. В тестах версии фиксированы в README.

## Подготовка
При первом запуске раскомментируйте строку `%pip` в следующей ячейке, выполните её и перезапустите ядро. Затем выполняйте ноутбук сверху вниз. Если зависимости уже установлены по requirements.txt, повторная установка не нужна. Не включайте кэш, фотографии и веса в сдачу.

In [ ]:
# Требуется интернет только для первого запуска. Установку выполните один раз в новом ядре.
# %pip install -q torch==2.5.1 torchvision==0.20.1 numpy==2.4.6 matplotlib==3.11.0 pillow==12.3.0 ipywidgets==8.1.9 transformers==4.51.3 tokenizers==0.21.1 huggingface-hub==0.30.2 safetensors==0.5.3
# После изменения пакетов перезапустите ядро и выполните ноутбук сверху вниз.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
import torch
if not torch.cuda.is_available(): torch.set_num_threads(min(4,torch.get_num_threads()))

In [ ]:
DATA_MANIFEST = [{'id': 'astronaut', 'filename': 'astronaut.png', 'sha256': '88431cd9653ccd539741b555fb0a46b61558b301d4110412b5bc28b5e3ea6cb5', 'url': 'https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/astronaut.png', 'credit': 'NASA, public domain'}, {'id': 'camera', 'filename': 'camera.png', 'sha256': 'b0793d2adda0fa6ae899c03989482bff9a42d3d5690fc7e3648f2795d730c23a', 'url': 'https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/camera.png', 'credit': 'Lav Varshney, CC0'}, {'id': 'coffee', 'filename': 'coffee.png', 'sha256': 'cc02f8ca188b167c775a7101b5d767d1e71792cf762c33d6fa15a4599b5a8de7', 'url': 'https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/coffee.png', 'credit': 'Rachel Michetti, CC0'}, {'id': 'chelsea', 'filename': 'chelsea.png', 'sha256': '596aa1e7cb875eb79f437e310381d26b338a81c2da23439704a73c4651e8c4bb', 'url': 'https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/chelsea.png', 'credit': 'Stefan van der Walt, CC0'}, {'id': 'rocket', 'filename': 'rocket.jpg', 'sha256': 'c2dd0de7c538df8d111e479619b129464d0269d0ae5fd18ca91d33a7fdfea95c', 'url': 'https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/rocket.jpg', 'credit': 'SpaceX, public domain'}, {'id': 'coins', 'filename': 'coins.png', 'sha256': 'f8d773fc9cfa6f4d8e5942dc34d0a0788fcaed2a4fefbbed0aef5398d7ef4cba', 'url': 'https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/coins.png', 'credit': 'Brooklyn Museum, no known copyright restrictions'}, {'id': 'horse', 'filename': 'horse.png', 'sha256': 'c7fb60789fe394c485f842291ea3b21e50d140f39d6dcb5fb9917cc178225455', 'url': 'https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/horse.png', 'credit': 'Andreas Preuss, CC0; illustration'}, {'id': 'hubble_deep_field', 'filename': 'hubble_deep_field.jpg', 'sha256': '3a19c5dd8a927a9334bb1229a6d63711b1c0c767fb27e2286e7c84a3e2c2f5f4', 'url': 'https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/hubble_deep_field.jpg', 'credit': 'NASA, public domain'}]
MODEL_MANIFEST = {'clip': {'repo': 'openai/clip-vit-base-patch32', 'revision': '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268', 'files': [{'name': 'config.json', 'url': 'https://huggingface.co/openai/clip-vit-base-patch32/resolve/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json', 'sha256': 'b575ef3c36f2a057fa19e221650105052d61cc9c1a972ec15019c6261ec98770', 'size': 4186}, {'name': 'merges.txt', 'url': 'https://huggingface.co/openai/clip-vit-base-patch32/resolve/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/merges.txt', 'sha256': 'f526393189112391ce6f9795d4695f704121ce452c3aad1f5335cc41337eba85', 'size': 524657}, {'name': 'preprocessor_config.json', 'url': 'https://huggingface.co/openai/clip-vit-base-patch32/resolve/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/preprocessor_config.json', 'sha256': '910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd', 'size': 316}, {'name': 'tokenizer.json', 'url': 'https://huggingface.co/openai/clip-vit-base-patch32/resolve/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/tokenizer.json', 'sha256': 'b556ac8c99757ffb677208af34bc8c6721572114111a6e0aaf5fa69ff0b8d842', 'size': 2224041}, {'name': 'tokenizer_config.json', 'url': 'https://huggingface.co/openai/clip-vit-base-patch32/resolve/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/tokenizer_config.json', 'sha256': '34b7336e4bee12e0a9730eaf5189f582ef3c3eea5027f65730e5717256755aad', 'size': 592}, {'name': 'vocab.json', 'url': 'https://huggingface.co/openai/clip-vit-base-patch32/resolve/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/vocab.json', 'sha256': '5047b556ce86ccaf6aa22b3ffccfc52d391ea4accdab9c2f2407da5b742d4363', 'size': 862328}, {'name': 'pytorch_model.bin', 'url': 'https://huggingface.co/openai/clip-vit-base-patch32/resolve/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/pytorch_model.bin', 'sha256': 'a63082132ba4f97a80bea76823f544493bffa8082296d62d71581a4feff1576f', 'size': 605247071}]}, 'dino': {'repo': 'IDEA-Research/grounding-dino-tiny', 'revision': 'a2bb814dd30d776dcf7e30523b00659f4f141c71', 'files': [{'name': 'added_tokens.json', 'url': 'https://huggingface.co/IDEA-Research/grounding-dino-tiny/resolve/a2bb814dd30d776dcf7e30523b00659f4f141c71/added_tokens.json', 'sha256': '909e96cb32d92ce728a01bc99850cbba26196d74115c17ebeb019275412588f2', 'size': 82}, {'name': 'config.json', 'url': 'https://huggingface.co/IDEA-Research/grounding-dino-tiny/resolve/a2bb814dd30d776dcf7e30523b00659f4f141c71/config.json', 'sha256': 'eec82c5ab66e16df12a9a212e68ac011779927c2536cf9078658e35d85f0c67a', 'size': 1644}, {'name': 'model.safetensors', 'url': 'https://huggingface.co/IDEA-Research/grounding-dino-tiny/resolve/a2bb814dd30d776dcf7e30523b00659f4f141c71/model.safetensors', 'sha256': '1a2412ef99bd74bcd3c2a246fa1e48581f8889a1300c9051974741314fc042f3', 'size': 689359096}, {'name': 'preprocessor_config.json', 'url': 'https://huggingface.co/IDEA-Research/grounding-dino-tiny/resolve/a2bb814dd30d776dcf7e30523b00659f4f141c71/preprocessor_config.json', 'sha256': '8454179ba95e2ad22947835aad7b45862a601fc0055ab88bf1ee70892d3aea60', 'size': 457}, {'name': 'tokenizer.json', 'url': 'https://huggingface.co/IDEA-Research/grounding-dino-tiny/resolve/a2bb814dd30d776dcf7e30523b00659f4f141c71/tokenizer.json', 'sha256': 'd241a60d5e8f04cc1b2b3e9ef7a4921b27bf526d9f6050ab90f9267a1f9e5c66', 'size': 711396}, {'name': 'tokenizer_config.json', 'url': 'https://huggingface.co/IDEA-Research/grounding-dino-tiny/resolve/a2bb814dd30d776dcf7e30523b00659f4f141c71/tokenizer_config.json', 'sha256': 'd40ab645b68211910b9170d22433d43186a6ec8ee6fd10ba170524b25bf4fb56', 'size': 1237}, {'name': 'vocab.txt', 'url': 'https://huggingface.co/IDEA-Research/grounding-dino-tiny/resolve/a2bb814dd30d776dcf7e30523b00659f4f141c71/vocab.txt', 'sha256': '07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3', 'size': 231508}]}, 'sam': {'repo': 'facebook/sam-vit-base', 'revision': '70c1a07f894ebb5b307fd9eaaee97b9dfc16068f', 'files': [{'name': 'config.json', 'url': 'https://huggingface.co/facebook/sam-vit-base/resolve/70c1a07f894ebb5b307fd9eaaee97b9dfc16068f/config.json', 'sha256': '5ebd0d8643b486f3a716bf17c2a15531eb818b2b96ff0c6c5dcc88fa015161af', 'size': 6566}, {'name': 'model.safetensors', 'url': 'https://huggingface.co/facebook/sam-vit-base/resolve/70c1a07f894ebb5b307fd9eaaee97b9dfc16068f/model.safetensors', 'sha256': '892c410e496344e527255ccdcb2cb7244a609acb5389c7c4fdba1288f861c579', 'size': 374979480}, {'name': 'preprocessor_config.json', 'url': 'https://huggingface.co/facebook/sam-vit-base/resolve/70c1a07f894ebb5b307fd9eaaee97b9dfc16068f/preprocessor_config.json', 'sha256': '225545a743c654e3c495ec6f545a0eaba57c8ba3fbbd8483b3cb1c0fc58db517', 'size': 466}]}}
"""Public helpers; model inference is real, synthetic teaching data are labelled."""
from pathlib import Path
import os, json, hashlib, time, zipfile, urllib.request, urllib.error
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

CACHE = Path(os.environ.get('VISUAL_LAB_CACHE', str(Path.home()/'.cache'/'applied-ai-visual')))
CACHE.mkdir(parents=True, exist_ok=True)

def select_device():
    """Use a PyTorch-visible CUDA GPU by default, with an explicit CPU override."""
    import torch
    requested=os.environ.get('VISUAL_LAB_DEVICE','auto').strip().lower()
    if requested not in ('auto','cuda','cpu'):
        raise ValueError("VISUAL_LAB_DEVICE must be 'auto', 'cuda', or 'cpu'")
    if requested=='cuda' and not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable to PyTorch; check the driver and CUDA-enabled PyTorch installation")
    device='cuda' if requested=='cuda' or (requested=='auto' and torch.cuda.is_available()) else 'cpu'
    if device=='cuda':
        print('Устройство для CLIP, Grounding DINO и SAM: CUDA — '+torch.cuda.get_device_name(0))
    else:
        print('Устройство для CLIP, Grounding DINO и SAM: CPU. Для GPU проверьте torch.cuda.is_available().')
    return device

DEVICE = select_device()

# DATA_MANIFEST and MODEL_MANIFEST are embedded above this cell.
def sha256(path):
    digest=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): digest.update(chunk)
    return digest.hexdigest()

def fetch(url, path, expected):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path)==expected: return path
    part=path.with_suffix(path.suffix+'.part')
    print('Загружаю',path.name,'— размер и ход загрузки показаны ниже.',flush=True)
    try:
        with urllib.request.urlopen(url,timeout=30) as response, part.open('wb') as output:
            total=response.headers.get('Content-Length')
            total=int(total) if total and total.isdigit() else None
            received=0;last_report=time.monotonic();next_report=10*1024*1024
            while True:
                chunk=response.read(1024*1024)
                if not chunk: break
                output.write(chunk);received+=len(chunk)
                now=time.monotonic()
                if received>=next_report or now-last_report>=10:
                    progress=f'{received/2**20:.1f} MiB'
                    if total: progress+=f' из {total/2**20:.1f} MiB'
                    print(path.name+': '+progress,flush=True)
                    next_report=received+10*1024*1024;last_report=now
        if sha256(part)!=expected: raise ValueError('Неверная контрольная сумма SHA-256: '+path.name)
        part.replace(path)
        print(path.name+': загружен и проверен.',flush=True)
    except (urllib.error.URLError,TimeoutError) as exc:
        part.unlink(missing_ok=True)
        raise RuntimeError(f'Не удалось загрузить {path.name}: {exc}. Проверьте интернет и повторите запуск.') from exc
    except Exception:
        part.unlink(missing_ok=True)
        raise
    return path

def gallery():
    return [Image.open(fetch(r['url'],CACHE/'gallery'/r['filename'],r['sha256'])).convert('RGB') for r in DATA_MANIFEST]

class FrozenCLIP:
    def __init__(self,device=None):
        import torch
        from transformers import CLIPModel,CLIPProcessor
        self.torch=torch; self.device=DEVICE if device is None else device
        path=model_files('clip')
        self.processor=CLIPProcessor.from_pretrained(path,local_files_only=True)
        self.model=CLIPModel.from_pretrained(path,local_files_only=True).eval().to(self.device)
    def images(self, images):
        t=time.perf_counter()
        x=self.processor(images=images,return_tensors='pt').to(self.device)
        with self.torch.inference_mode(): y=self.model.get_image_features(**x)
        if self.device.startswith('cuda'): self.torch.cuda.synchronize()
        self.last_seconds=time.perf_counter()-t
        return y.cpu().numpy().astype(np.float64)
    def texts(self,texts):
        x=self.processor(text=texts,padding=True,truncation=True,return_tensors='pt').to(self.device)
        with self.torch.inference_mode(): y=self.model.get_text_features(**x)
        return y.cpu().numpy().astype(np.float64)

def model_files(name):
    spec=MODEL_MANIFEST[name]
    target=CACHE/'models'/name
    for item in spec['files']: fetch(item['url'],target/item['name'],item['sha256'])
    return target

def unit(x):
    x=np.asarray(x,dtype=float)
    if not np.isfinite(x).all(): raise ValueError('Non-finite vector')
    scale=np.max(np.abs(x),axis=-1,keepdims=True,initial=0)
    scaled=np.divide(x,scale,out=np.zeros_like(x),where=scale>0)
    length=np.sqrt(np.sum(scaled*scaled,axis=-1,keepdims=True))
    return np.divide(scaled,length,out=np.zeros_like(scaled),where=length>0)

def softmax(x):
    x=np.asarray(x,float); e=np.exp(x-np.max(x,axis=-1,keepdims=True))
    return e/e.sum(axis=-1,keepdims=True)

def show_gallery(images, scores=None, order=None, title=''):
    order=np.arange(len(images)) if order is None else np.asarray(order)
    fig, axes=plt.subplots(1,len(order),figsize=(2.1*len(order),2.7),squeeze=False)
    for ax,j in zip(axes.flat,order):
        ax.imshow(images[j]);ax.axis('off')
        ax.set_title(DATA_MANIFEST[j]['id']+(f'\ncos={scores[j]:.3f}' if scores is not None else ''),fontsize=9)
    fig.suptitle(title);plt.tight_layout();plt.show()
    return fig

PENN_SHA='9095a9613c95586f1c7f2a327d454833d16e0f5e17e5f83d35027ffd315b48e2'
PENN_IDS=['FudanPed00001','FudanPed00002','FudanPed00003','FudanPed00004']
def penn_sample(index=0):
    archive=CACHE/'PennFudanPed.zip'
    if not (archive.is_file() and sha256(archive)==PENN_SHA):
        detection_root=os.environ.get('DETECTION_LAB_ROOT')
        shared=Path(detection_root)/'PennFudanPed.zip' if detection_root else None
        if shared is not None and shared.is_file() and sha256(shared)==PENN_SHA:
            archive=shared
            print('Использую проверенный архив Penn–Fudan из кэша курса.',flush=True)
        else:
            archive=fetch('https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip',archive,PENN_SHA)
    # Read only the requested members; no unsafe archive extraction.
    name=PENN_IDS[index]
    with zipfile.ZipFile(archive) as z:
        with z.open('PennFudanPed/PNGImages/'+name+'.png') as f: image=Image.open(f).convert('RGB')
        with z.open('PennFudanPed/PedMasks/'+name+'_mask.png') as f: ids=np.array(Image.open(f))
    masks=np.array([ids==i for i in np.unique(ids) if i!=0])
    boxes=[]
    for mask in masks:
        ys,xs=np.where(mask);boxes.append([xs.min(),ys.min(),xs.max()+1,ys.max()+1])
    return image,np.array(boxes,float),masks

class TextToMask:
    def __init__(self,device=None):
        import torch
        from transformers import AutoProcessor,AutoModelForZeroShotObjectDetection
        self.torch=torch;self.device=DEVICE if device is None else device
        dino_files=model_files('dino')
        self.dp=AutoProcessor.from_pretrained(dino_files,local_files_only=True)
        self.dm=AutoModelForZeroShotObjectDetection.from_pretrained(dino_files,local_files_only=True).eval().to(self.device)
        self.sp=None;self.sm=None
    def detect(self,image,text='a person.',box_threshold=0.25,text_threshold=0.25):
        t=time.perf_counter()
        key=(hashlib.sha256(image.tobytes()).hexdigest(),text)
        if getattr(self,'_key',None)!=key:
            x=self.dp(images=image,text=text,return_tensors='pt').to(self.device)
            with self.torch.inference_mode(): y=self.dm(**x)
            self._key,self._raw=key,(x,y)
        else: x,y=self._raw
        r=self.dp.post_process_grounded_object_detection(y,x.input_ids,box_threshold=box_threshold,text_threshold=text_threshold,target_sizes=[image.size[::-1]])[0]
        if self.device.startswith('cuda'): self.torch.cuda.synchronize()
        return dict(boxes=r['boxes'].cpu().numpy(),scores=r['scores'].cpu().numpy(),labels=r['text_labels'],seconds=time.perf_counter()-t)
    def segment(self,image,boxes):
        if len(boxes)==0: return np.empty((0,image.height,image.width),bool),np.empty(0),0.
        if self.sm is None:
            from transformers import SamModel,SamProcessor
            sam_files=model_files('sam')
            self.sp=SamProcessor.from_pretrained(sam_files,local_files_only=True)
            self.sm=SamModel.from_pretrained(sam_files,local_files_only=True).eval().to(self.device)
        t=time.perf_counter()
        x=self.sp(images=image,input_boxes=[np.asarray(boxes).tolist()],return_tensors='pt').to(self.device)
        # The image is fixed while Stand 7 changes box prompts. Reuse only its vision embedding.
        image_key=(hashlib.sha256(image.tobytes()).digest(),image.size,image.mode)
        with self.torch.inference_mode():
            if getattr(self,'_sam_image_key',None)!=image_key:
                self._sam_image_embeddings=self.sm.get_image_embeddings(x['pixel_values'])
                self._sam_image_key=image_key
            y=self.sm(image_embeddings=self._sam_image_embeddings,input_boxes=x['input_boxes'],multimask_output=False)
        # transformers 4.51.3 interpolates on the masks' device; transfer only final masks.
        masks=self.sp.image_processor.post_process_masks(y.pred_masks,x['original_sizes'].cpu(),x['reshaped_input_sizes'].cpu())[0][:,0].cpu().numpy().astype(bool)
        if self.device.startswith('cuda'): self.torch.cuda.synchronize()
        return masks,y.iou_scores[0,:,0].cpu().numpy(),time.perf_counter()-t

def box_iou(a,b):
    a=np.asarray(a,float).reshape(-1,4);b=np.asarray(b,float).reshape(-1,4)
    lo=np.maximum(a[:,None,:2],b[None,:,:2]);hi=np.minimum(a[:,None,2:],b[None,:,2:])
    inter=np.prod(np.maximum(0,hi-lo),axis=-1)
    aa=np.prod(np.maximum(0,a[:,2:]-a[:,:2]),axis=-1)
    bb=np.prod(np.maximum(0,b[:,2:]-b[:,:2]),axis=-1)
    return inter/np.maximum(aa[:,None]+bb[None,:]-inter,1e-12)

def mask_iou(a,b):
    a=np.asarray(a,bool);b=np.asarray(b,bool)
    inter=np.logical_and(a,b).sum();union=np.logical_or(a,b).sum()
    return float(inter/union) if union else 1.

def draw_pipeline(image,boxes,masks=None,gt_boxes=None,title=''):
    fig,ax=plt.subplots(figsize=(8,5));ax.imshow(image)
    for i,b in enumerate(boxes):
        ax.add_patch(Rectangle(b[:2],b[2]-b[0],b[3]-b[1],fill=False,edgecolor='lime',linewidth=2))
        ax.text(b[0],b[1],str(i),color='black',bbox=dict(facecolor='lime',alpha=.8))
        if masks is not None and i<len(masks):
            overlay=np.zeros((*masks[i].shape,4));overlay[masks[i]]=[0,1,0,.25];ax.imshow(overlay)
    if gt_boxes is not None:
        for b in gt_boxes: ax.add_patch(Rectangle(b[:2],b[2]-b[0],b[3]-b[1],fill=False,edgecolor='magenta',linestyle='--'))
    ax.set_title(title);ax.axis('off');plt.show();return fig

# Fixed gallery; split concerns unseen wording on the SAME images, not unseen images.
VAL_QUERIES=['astronaut','photographer with a camera','cup of coffee','cat','rocket launch','coins','horse silhouette','galaxies']
TEST_QUERIES=['a woman wearing a space suit','a person taking a photograph outdoors','latte art on a wooden table','a close up of a tabby cat','a spacecraft lifting off at night','several ancient round metal coins','a black and white drawing of a horse','a deep view of distant stars and galaxies']
NEGATIVE_VAL=['a red fire truck','a tennis match','a bowl of strawberries','a snowy mountain']
NEGATIVE_TEST=['a blue bicycle','a violin on a chair','a penguin on ice','a crowded city bus']
TEMPLATES=['{}','a photo of {}','a close-up photo of {}']

def model_versions():
    import platform,torch,transformers
    return dict(python=platform.python_version(),torch=torch.__version__,transformers=transformers.__version__,device=DEVICE,models={k:v['revision'] for k,v in MODEL_MANIFEST.items()})


# Дополнительная самостоятельная работа · выберите один маршрут

Этот ноутбук полностью автономен: не импортирует решения основной работы и повторно загружает только
общий проверенный кэш. Выберите **одну** исследовательскую задачу на 60–120 минут.
Результат — гипотеза, заранее записанный протокол, собственная проверка, таблица и анализ ошибок.
Максимальное качество не является критерием. Приведённые ячейки только готовят baseline.

In [ ]:
images=gallery();clip=FrozenCLIP();image_features=clip.images(images)
show_gallery(images,title='Общий baseline для выбранного маршрута')

## A · Язык запроса как контролируемое изменение

OpenAI CLIP не валидирован авторами для произвольного многоязычного применения. Проверьте восемь пар
семантически сопоставимых английских/русских формулировок. Это стресс-тест конкретной модели, не рейтинг языков.

До запуска составьте свою таблицу переводов и минимум два неоднозначных случая; заморозьте её.
Сравните rank соответствующей картинки, margin и совпадение top-3. Подготовьте ещё четыре пары test только
после фиксации метода, до запуска их кодирования. Не подбирайте перевод по scores. Объясните ограничения
маленькой фиксированной галереи и хотя бы одну ошибку, связанную с смыслом, а не только с языком.

In [ ]:
english=['astronaut','photographer','coffee','cat','rocket','coins','horse silhouette','galaxies']
russian=['астронавт','фотограф','кофе','кот','ракета','монеты','силуэт лошади','галактики']
# Замените и зафиксируйте пары до опыта, обоснуйте семантическое соответствие.
language_baseline={name:unit(clip.texts(queries))@unit(image_features).T for name,queries in [('en',english),('ru',russian)]}
# TODO: собственный оценщик парных rank/margin/top-3 и ожидаемый контрольный пример.

## B · Цена сжатия визуального индекса

Сформулируйте гипотезу: равномерное квантование нормированных image-векторов 8/4/2 бит может нарушать порядок
похожих кандидатов. Текстовые векторы и галерея одинаковы. Реализуйте квантование с явно описанным масштабом,
обратное преобразование и повторную нормализацию; контрольные случаи: нулевой, крайние ±1, почти равные scores.

Сравните память **самих массивов** и память реального упакованного представления, cosine error, top-1 agreement
и top-3 overlap на заранее заданных val/test формулировках. Не заявляйте ускорение поиска по восьми векторам.
Отделите сжатие float32→uint8 от упаковки четырёх/двух бит: uint8 на каждый элемент не даёт 4-битного хранения.

In [ ]:
reference_index=unit(image_features).astype(np.float32)
reference_queries=unit(clip.texts(VAL_QUERIES)).astype(np.float32)
reference_scores=reference_queries@reference_index.T
print('Float32 index bytes:',reference_index.nbytes)
# TODO: quantize/dequantize, собственные assert, реальная упаковка, общая таблица 8/4/2 бит на val.
# После выбора метода/масштаба запишите решение в JSON, затем отдельной ячейкой
# кодируйте TEST_QUERIES и сравните один раз с зафиксированным float32 baseline.

## C · Отдельный test на новых изображениях

Текущий основной test проверяет формулировки на прежних изображениях. Исправьте именно это ограничение:
соберите 12–20 собственных или разрешённых к использованию новых изображений с distractor-кадрами и минимум
четырьмя запросами, имеющими несколько relevant. Укажите автора, лицензию и SHA-256 каждого файла.

До инференса дважды проверьте бинарную query×image разметку, опишите неоднозначные пары, сохраните разметку
и фиксированный split. Выберите шаблон/порог только на val; один раз оцените test новых изображений.
Сравните с исходным baseline при тех же frozen весах и одинаковом оценщике. Отсутствие улучшения допустимо.
Сдайте манифест/разметку/код, не распространяйте изображения без соответствующего права.

In [ ]:
# TODO: ваш manifest с url/credit/license/sha256, релевантность, split и оценщик.
# Весь общий код fetch(), FrozenCLIP() и unit() уже доступен в этом ноутбуке.
NEW_DATA_MANIFEST=[]

## Формат дополнительной сдачи

Отдельный выполненный ноутбук + 1–2 страницы: гипотеза и её предварительная фиксация, данные и права,
контрольный пример, таблица, два трудных случая, вывод с конкретным числом и ограничение.
Эта часть оценивается отдельно (до 10 бонусных баллов); не заменяет обязательный эксперимент.